In [ ]:
!pip install -q kaggle
!kaggle datasets download -d zalando-research/fashionmnist
!unzip fashionmnist.zip -d fashionmnist/
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import tensorflow as tf
import os
import kagglehub
from matplotlib import pyplot as plt
from google.colab import drive
drive.mount('/content/drive')

ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run


In [ ]:
train_df = pd.read_csv('fashionmnist/fashion-mnist_train.csv')
test_df = pd.read_csv('fashionmnist/fashion-mnist_test.csv')

In [ ]:
def image_from_row(row, df=train_df):
    return 2*df.iloc[row, 1:].values.reshape(28,28,1)/255 -1

def images_from_df(indices, df=train_df):
    return 2*df.iloc[indices, 1:].values.reshape(len(indices), 28,28,1)/255 -1

In [ ]:
def residual_block(x, filters):
    shortcut = x

    if x.shape[-1] != filters:
        shortcut = layers.Conv2D(
            filters,
            1,
            padding="same"
        )(shortcut)

    x = layers.Conv2D(filters, 3, padding="same")(x)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Conv2D(filters, 3, padding="same")(x)

    x = layers.Add()([x, shortcut])
    x = layers.LeakyReLU(0.2)(x)

    return x

In [ ]:
from tensorflow.keras import models, layers
def criar_critico():
  inputs = layers.Input((28,28,1))
  x = layers.Conv2D(64, 3, padding="same")(inputs)
  x = residual_block(x, 64)
  x = residual_block(x, 128)
  x = residual_block(x, 256)
  x = layers.GlobalAveragePooling2D()(x)
  x = layers.Dense(128)(x)
  outputs = layers.Dense(1)(x)
  return models.Model(inputs, outputs)
criar_critico().summary()

In [ ]:
GENERATOR_LATENT_DIM = 512
def criar_gerador():
  inputs = layers.Input((GENERATOR_LATENT_DIM,))
  x = layers.Dense(512)(inputs)
  x = layers.Reshape((4,4,32))(x)
  x = residual_block(x, 128)
  x = layers.UpSampling2D((2,2))(x)
  x = layers.BatchNormalization()(x)
  x = layers.UpSampling2D((2,2))(x)
  x = residual_block(x, 64)
  x = layers.UpSampling2D((2,2))(x)
  x = layers.BatchNormalization()(x)
  x = residual_block(x, 32)
  x = residual_block(x, 16)
  outputs = layers.Conv2D(1, (5, 5), activation='tanh')(x)
  return  models.Model(inputs, outputs)
criar_gerador().summary()

In [ ]:
train_images = train_df.iloc[:, 1:].values.astype("float32")
train_images = train_images.reshape(-1, 28, 28, 1)

train_images = train_images / 127.5 - 1
train_images = train_images.astype(np.float32)
def images_from_array(indices):
    return train_images[indices]


In [ ]:
@tf.function
def train_critic(
    imagens_reais,
    optimizer_critico,
    batch_size,
    lambda_gp
  ):
  print("Entrou no train_critic")
  batch = tf.shape(imagens_reais)[0]
  z = tf.random.normal((batch, GENERATOR_LATENT_DIM), dtype=tf.float32)
  imagens_geradas = tf.stop_gradient(
      gerador(z, training=False)
  )
  with tf.GradientTape() as tape:
      y_pred_gerado = critico(imagens_geradas, training=True)
      # print("do y_pred_gerado contain nan?", tf.reduce_any(tf.math.is_nan(y_pred_gerado)))
      y_pred_real = critico(imagens_reais, training=True)
      # tf.print(
      #     "critic output",
      #     tf.reduce_min(y_pred_real),
      #     tf.reduce_max(y_pred_real)
      # )
      # print("do y_pred_real contain nan?", tf.reduce_any(tf.math.is_nan(y_pred_real)))
      # print("wasserstein =", tf.reduce_mean(y_pred_gerado) - tf.reduce_mean(y_pred_real))
      epsilon = tf.random.uniform(
          [batch,1,1,1], dtype=tf.float32
      )
      # print("epsilon", epsilon)
      x_hat = (
          epsilon*imagens_reais
          +
          (one-epsilon)*imagens_geradas
      )
      with tf.GradientTape() as gp_tape:
          gp_tape.watch(x_hat)

          pred = critico(x_hat)
      grad = gp_tape.gradient(pred, x_hat)
      del gp_tape
      norm = tf.sqrt(
          tf.reduce_sum(
              tf.square(grad),
              axis=[1,2,3]
          )
      )
      gp = tf.reduce_mean(
          (norm - one) ** 2
      )
      loss_critico = tf.reduce_mean(y_pred_gerado) - tf.reduce_mean(y_pred_real) + lambda_gp*gp
      # print("loss =", loss_critico)
      # print("gp =", gp)
      # print("norm mean =", tf.reduce_mean(norm))
      # print("norm max =", tf.reduce_max(norm))
  grads_critico = tape.gradient(loss_critico, critico.trainable_variables)
  optimizer_critico.apply_gradients(
      zip(grads_critico, critico.trainable_variables)
  )
  # for v in critico.trainable_variables:
  #   if tf.reduce_any(tf.math.is_nan(v)):
  #       print("Critico corrompeu:", v.name)
  #       raise RuntimeError("NaN no crítico")

@tf.function
def train_generator(
    optimizer_gerador,
    batch_size,
    lambda_gp,
    tamanho_batch_atual
):
  batch = tf.shape(imagens_reais)[0]
  z = tf.random.normal((tamanho_batch_atual, GENERATOR_LATENT_DIM), dtype=tf.float32)
  with tf.GradientTape() as tape:
      imagens_geradas = gerador(z, training=True)
      y_pred_gerado = critico(imagens_geradas, training=False)
      loss_gerador = -tf.reduce_mean(y_pred_gerado)

  grads_gerador = tape.gradient(loss_gerador, gerador.trainable_variables)
  optimizer_gerador.apply_gradients(
      zip(grads_gerador, gerador.trainable_variables)
  )
  for v in gerador.trainable_variables:
    if tf.reduce_any(tf.math.is_nan(v)):
        print("Gerador corrompeu:", v.name)
        raise RuntimeError("NaN no gerador")


In [ ]:
import math
from tqdm.notebook import tqdm
from keras.losses import BinaryCrossentropy
from keras.optimizers import Adam
import pickle
import time


batch_size = 200
epochs = 50
lambda_gp = 10
num_batches = int(math.ceil(train_df.shape[0]/batch_size))
n_critic = 5
save_interval = 128
dataset = tf.data.Dataset.from_tensor_slices(train_images)
dataset = dataset.shuffle(
    len(train_images)
).batch(
    batch_size
).prefetch(
    tf.data.AUTOTUNE
)

with tf.device('/device:GPU:0'):
  gerador = criar_gerador()
  critico = criar_critico()
  optimizer_gerador = Adam(learning_rate=0.0001, beta_1=0.0, beta_2=0.9)
  optimizer_critico = Adam(learning_rate=0.0001, beta_1=0.0, beta_2=0.9)
  # WARMING UP THE OPTIMIZERS
  dummy = tf.zeros((1, 28, 28, 1))

  with tf.GradientTape() as tape:
      out = critico(dummy)

  grads = tape.gradient(out, critico.trainable_variables)

  optimizer_critico.apply_gradients(
      zip(grads, critico.trainable_variables)
  )
  z = tf.random.normal((tamanho_batch_atual, GENERATOR_LATENT_DIM), dtype=tf.float32)
  with tf.GradientTape() as tape:
      imagens_geradas = gerador(z, training=True)
      y_pred_gerado = critico(imagens_geradas, training=False)
      loss_gerador = -tf.reduce_mean(y_pred_gerado)

  grads_gerador = tape.gradient(loss_gerador, gerador.trainable_variables)
  optimizer_gerador.apply_gradients(
      zip(grads_gerador, gerador.trainable_variables)
  )

  # CHECKPOINTS DEFINITION AND RESTORATION
  ckpt = tf.train.Checkpoint(
      gerador=gerador,
      critico=critico,
      optimizer_gerador=optimizer_gerador,
      optimizer_critico=optimizer_critico,
      epoch=tf.Variable(0),
      batch=tf.Variable(0)
  )

  manager = tf.train.CheckpointManager(
      ckpt,
      "drive/MyDrive/machine_learning_systems_output/Fashion_MNIST/checkpoints",
      max_to_keep=1
  )
  print(manager.latest_checkpoint)
  ckpt.restore(manager.latest_checkpoint)

  initial_epoch = 0
  initial_batch = 0
  if manager.latest_checkpoint:
      print("Checkpoint carregado!")
      print("Época:", int(ckpt.epoch))
      print("Batch:", int(ckpt.batch))
      initial_epoch =  int(ckpt.epoch.numpy())
      initial_batch = int(ckpt.batch.numpy())
  else:
      print("Treinamento do zero.")
  one = tf.constant(1.0, dtype=tf.float32)
  for epoch in tqdm(
      range(initial_epoch, epochs),
      initial=initial_epoch,
      total=epochs,
      desc="epoch"
  ):
      start_batch = initial_batch if epoch == initial_epoch else 0

      for batch_index, imagens_reais in tqdm(
          enumerate(dataset),
          initial=start_batch,
          total=num_batches,
          desc="batch",
          leave=False
      ):
          if batch_index < start_batch:
              continue

          tamanho_batch_atual = imagens_reais.shape[0]
          ruido = tf.random.normal(
              imagens_reais.shape,
              stddev=0.05,
              dtype=imagens_reais.dtype
          )
          imagens_reais += ruido
          imagens_reais = tf.clip_by_value(
              imagens_reais,
              -1.0,
              1.0
          )
          for _ in range(n_critic):
            train_critic(
              imagens_reais,
              optimizer_critico,
              batch_size,
              lambda_gp
            )
          train_generator(
              optimizer_gerador,
              batch_size,
              lambda_gp,
              tamanho_batch_atual
          )
          if batch_index % save_interval == 0:
            if batch_index == num_batches - 1:
              ckpt.epoch.assign(epoch + 1)
              ckpt.batch.assign(0)
            else:
              ckpt.epoch.assign(epoch)
              ckpt.batch.assign(batch_index + 1)
            save_path = manager.save()
            print("Checkpoint salvo em:", save_path)

**When the training needs to be stopped run the cells bellow to save the checkpoints and results on the repository**

In [ ]:
os.makedirs('drive/MyDrive/machine_learning_systems_output/Fashion_MNIST/generator_outputs', exist_ok=True)
z = z = tf.random.normal((10, GENERATOR_LATENT_DIM))
imagens = gerador(z)
print(imagens.shape)
print(imagens.dtype)

print(tf.reduce_min(imagens))
print(tf.reduce_max(imagens))

print(tf.math.reduce_any(tf.math.is_nan(imagens)))
print(tf.math.reduce_any(tf.math.is_inf(imagens)))
for i in range(10):
    plt.figure()
    plt.imsave(f'drive/MyDrive/machine_learning_systems_output/Fashion_MNIST/generator_outputs/{i}.png', tf.squeeze(imagens[i]), cmap="gray")

In [ ]:
plt.imshow(tf.squeeze(imagens[0].numpy), cmap="gray")
plt.colorbar()
plt.show()

In [ ]:
z = tf.random.normal((10, GENERATOR_LATENT_DIM))
imgs = gerador(z, training=False)

print(tf.reduce_min(imgs))
print(tf.reduce_max(imgs))
print(tf.math.reduce_any(tf.math.is_nan(imgs)))

In [ ]:
for v in gerador.trainable_variables:
    if tf.reduce_any(tf.math.is_nan(v)):
        print(v.name, "contém NaN")

In [ ]:
gerador = criar_gerador()
z = tf.random.normal((10, GENERATOR_LATENT_DIM))
imgs = gerador(z, training=False)

print(tf.reduce_min(imgs))
print(tf.reduce_max(imgs))
print(tf.math.reduce_any(tf.math.is_nan(imgs)))